In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')

# 1. Look at the shape (rows, columns)
print(df.shape)


(1470, 35)


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [3]:
# Check for duplicate rows
print(df.duplicated().sum())


0


In [4]:
# Check unique values for key categorical columns
for col in ['Attrition', 'Department', 'OverTime', 'BusinessTravel', 'Gender', 'MaritalStatus']:
    print(col, '->', df[col].unique())

Attrition -> ['Yes' 'No']
Department -> ['Sales' 'Research & Development' 'Human Resources']
OverTime -> ['Yes' 'No']
BusinessTravel -> ['Travel_Rarely' 'Travel_Frequently' 'Non-Travel']
Gender -> ['Female' 'Male']
MaritalStatus -> ['Single' 'Married' 'Divorced']


In [5]:
# Check for "constant" columns (same value in every row — useless for analysis)
for col in df.columns:
    if df[col].nunique() == 1:
        print(col, 'has only one unique value:', df[col].unique())

EmployeeCount has only one unique value: [1]
Over18 has only one unique value: ['Y']
StandardHours has only one unique value: [80]


In [6]:
# Make a copy so we never touch the raw df directly
df_clean = df.copy()

# Drop constant/useless columns
df_clean = df_clean.drop(columns=['EmployeeCount', 'Over18', 'StandardHours'])

# Also drop EmployeeNumber from analysis later (it's just an ID, useful only as a key, not for stats)
# We'll KEEP it for now since it's our unique identifier - just noting we won't use it in aggregations

print(df_clean.shape)

(1470, 32)


In [7]:
# Convert Attrition to a cleaner format for analysis (optional but good practice)
# Yes/No is fine for now - we'll use it directly in groupby operations

# Let's create one new helpful column: AttritionFlag (1/0) for easier math later
df_clean['AttritionFlag'] = df_clean['Attrition'].map({'Yes': 1, 'No': 0})

df_clean[['Attrition', 'AttritionFlag']].head()

,Attrition,AttritionFlag
0,Yes,1
1,No,0
2,Yes,1
3,No,0
4,No,0


In [8]:
# Save the cleaned dataset
df_clean.to_csv('data/cleaned/hr_attrition_cleaned.csv', index=False)
print("Saved successfully")

Saved successfully


In [9]:
attrition_rate = df_clean['AttritionFlag'].mean() * 100
print(f"Overall Attrition Rate: {attrition_rate:.2f}%")

Overall Attrition Rate: 16.12%


In [10]:
import pandas as pd

In [11]:
import pandas as pd
df_clean = pd.read_csv('data/cleaned/hr_attrition_cleaned.csv')
df_clean.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,AttritionFlag
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,1,0,8,0,1,6,4,0,5,1
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,4,1,10,3,3,10,7,1,7,0
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,2,0,7,3,3,0,0,0,0,1
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,3,0,8,3,3,8,7,3,0,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,4,1,6,3,3,2,2,2,2,0


In [12]:
attrition_rate = df_clean['AttritionFlag'].mean() * 100
print(f"Overall Attrition Rate: {attrition_rate:.2f}%")

Overall Attrition Rate: 16.12%


In [13]:
dept_attrition = df_clean.groupby('Department')['AttritionFlag'].mean() * 100
print(dept_attrition.sort_values(ascending=False))

Department
Sales                     20.627803
Human Resources           19.047619
Research & Development    13.839750
Name: AttritionFlag, dtype: float64


In [14]:
dept_overtime_attrition = df_clean.groupby(['Department', 'OverTime'])['AttritionFlag'].mean() * 100
print(dept_overtime_attrition.sort_values(ascending=False))

Department              OverTime
Sales                   Yes         37.500000
Human Resources         Yes         29.411765
Research & Development  Yes         27.306273
Human Resources         No          15.217391
Sales                   No          13.836478
Research & Development  No           8.550725
Name: AttritionFlag, dtype: float64


In [15]:
income_attrition = df_clean.groupby('Attrition')['MonthlyIncome'].mean()
print(income_attrition)

Attrition
No     6832.739659
Yes    4787.092827
Name: MonthlyIncome, dtype: float64


In [16]:
income_tenure = df_clean.groupby('Attrition')[['MonthlyIncome', 'YearsAtCompany', 'Age']].mean()
print(income_tenure)

           MonthlyIncome  YearsAtCompany        Age
Attrition                                          
No           6832.739659        7.369019  37.561233
Yes          4787.092827        5.130802  33.607595


In [17]:
# Bucket employees into tenure groups, then compare income within each bucket
df_clean['TenureGroup'] = pd.cut(df_clean['YearsAtCompany'], 
                                    bins=[-1, 2, 5, 10, 100], 
                                    labels=['0-2 yrs', '3-5 yrs', '6-10 yrs', '10+ yrs'])

income_by_tenure_attrition = df_clean.groupby(['TenureGroup', 'Attrition'])['MonthlyIncome'].mean()
print(income_by_tenure_attrition)

TenureGroup  Attrition
0-2 yrs      No            5260.979167
             Yes           3411.352941
3-5 yrs      No            5552.553476
             Yes           4195.133333
6-10 yrs     No            6701.513995
             Yes           5608.854545
10+ yrs      No           10848.597345
             Yes          11319.400000
Name: MonthlyIncome, dtype: float64


C:\Users\Bhavishya Tyagi\AppData\Local\Temp\ipykernel_1524\374140314.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  income_by_tenure_attrition = df_clean.groupby(['TenureGroup', 'Attrition'])['MonthlyIncome'].mean()


In [18]:
import sqlite3

# Create (or connect to) a SQLite database file
conn = sqlite3.connect('hr_attrition.db')

# Load our cleaned dataframe into a SQL table called 'employees'
df_clean.to_sql('employees', conn, if_exists='replace', index=False)

print("Table created successfully")

Table created successfully


In [19]:
query = """
SELECT COUNT(*) as total_employees
FROM employees
"""

result = pd.read_sql_query(query, conn)
print(result)

   total_employees
0             1470


In [20]:
query1 = """
SELECT 
    Department,
    COUNT(*) AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS employees_left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
        2
    ) AS attrition_rate_pct
FROM employees
GROUP BY Department
ORDER BY attrition_rate_pct DESC;
"""

result1 = pd.read_sql_query(query1, conn)
print(result1)

               Department  total_employees  employees_left  attrition_rate_pct
0                   Sales              446              92               20.63
1         Human Resources               63              12               19.05
2  Research & Development              961             133               13.84


In [21]:
SELECT 
    Department,
    OverTime,
    COUNT(*) AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS employees_left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
        2
    ) AS attrition_rate_pct
FROM employees
GROUP BY Department, OverTime
ORDER BY attrition_rate_pct DESC;

IndentationError: unexpected indent (2670273414.py, line 2)

In [ ]:
query2 = """
SELECT 
    Department,
    OverTime,
    COUNT(*) AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS employees_left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
        2
    ) AS attrition_rate_pct
FROM employees
GROUP BY Department, OverTime
ORDER BY attrition_rate_pct DESC;
"""

result2 = pd.read_sql_query(query2, conn)
print(result2)

In [ ]:
# Query 1: Attrition rate by department
query1_sql = """SELECT 
    Department,
    COUNT(*) AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS employees_left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
        2
    ) AS attrition_rate_pct
FROM employees
GROUP BY Department
ORDER BY attrition_rate_pct DESC;"""

with open('sql/attrition_by_department.sql', 'w') as f:
    f.write(query1_sql)

# Query 2: Attrition rate by department + overtime
query2_sql = """SELECT 
    Department,
    OverTime,
    COUNT(*) AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS employees_left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
        2
    ) AS attrition_rate_pct
FROM employees
GROUP BY Department, OverTime
ORDER BY attrition_rate_pct DESC;"""

with open('sql/attrition_by_department_overtime.sql', 'w') as f:
    f.write(query2_sql)

# Query 3: Job role attrition ranked within each department (window function)
query3_sql = """SELECT 
    Department,
    JobRole,
    total_employees,
    employees_left,
    attrition_rate_pct,
    RANK() OVER (PARTITION BY Department ORDER BY attrition_rate_pct DESC) AS rank_within_dept
FROM (
    SELECT 
        Department,
        JobRole,
        COUNT(*) AS total_employees,
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS employees_left,
        ROUND(
            SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
            2
        ) AS attrition_rate_pct
    FROM employees
    GROUP BY Department, JobRole
) AS role_attrition
ORDER BY Department, rank_within_dept;"""

with open('sql/attrition_by_role_ranked.sql', 'w') as f:
    f.write(query3_sql)

print("All 3 queries saved successfully")

In [ ]:
result3 = pd.read_sql_query(query3_sql, conn)
print(result3)

In [ ]:
query4_sql = """WITH role_attrition AS (
    SELECT 
        Department,
        JobRole,
        OverTime,
        COUNT(*) AS total_employees,
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS employees_left,
        ROUND(
            SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
            2
        ) AS attrition_rate_pct
    FROM employees
    WHERE JobRole = 'Sales Representative'
    GROUP BY Department, JobRole, OverTime
)
SELECT *
FROM role_attrition
ORDER BY attrition_rate_pct DESC;"""

result4 = pd.read_sql_query(query4_sql, conn)
print(result4)

with open('sql/sales_rep_overtime_hypothesis.sql', 'w') as f:
    f.write(query4_sql)

In [ ]:
cost_of_attrition = (df_clean[df_clean['Attrition'] == 'Yes']['MonthlyIncome'] * 12 * 0.5).sum()
print(f"Estimated Annual Cost of Attrition: ${cost_of_attrition:,.2f}")"

In [ ]:
cost_of_attrition = (df_clean[df_clean['Attrition'] == 'Yes']['MonthlyIncome'] * 12 * 0.5).sum()
print(f"Estimated Annual Cost of Attrition: ${cost_of_attrition:,.2f}")

In [ ]:
high_risk_cost = (df_clean[
    (df_clean['JobRole'] == 'Sales Representative') & 
    (df_clean['OverTime'] == 'Yes') & 
    (df_clean['Attrition'] == 'Yes')
]['MonthlyIncome'] * 12 * 0.5).sum()

print(f"Cost attributable to Sales Rep + Overtime group: ${high_risk_cost:,.2f}")
print(f"% of total attrition cost from this group: {high_risk_cost/cost_of_attrition*100:.1f}%")

In [23]:
cost_by_dept = (df_clean[df_clean['Attrition'] == 'Yes']
                 .groupby('Department')['MonthlyIncome']
                 .apply(lambda x: (x * 12 * 0.5).sum()))
print(cost_by_dept.sort_values(ascending=False))

Department
Research & Development    3278244.0
Sales                     3261468.0
Human Resources            267534.0
Name: MonthlyIncome, dtype: float64


In [22]:
df_clean.to_excel('excel/hr_attrition_for_excel.xlsx', index=False, sheet_name='RawData')
print("Exported to Excel")

Exported to Excel
